In [ ]:
import os
import logging
import numpy as np
import nest_asyncio
import networkx as nx
import random
from IPython.display import HTML, display

In [2]:
from dotenv import load_dotenv
from openai import AzureOpenAI
from pathlib import Path

In [3]:
from lightrag import LightRAG, QueryParam
from lightrag.utils import EmbeddingFunc
from lightrag.kg.shared_storage import initialize_pipeline_status

In [26]:
from pyvis.network import Network

In [4]:
logging.basicConfig(level=logging.INFO)

In [5]:
load_dotenv()

True

In [13]:
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
AZURE_OPENAI_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")

AZURE_EMBEDDING_DEPLOYMENT = os.getenv("AZURE_EMBEDDING_DEPLOYMENT")
AZURE_EMBEDDING_API_VERSION = os.getenv("AZURE_EMBEDDING_API_VERSION")

DATABASE_FILES_DIR = Path("database_files")
WORKING_DIR = Path("working_dir")

EMBEDDING_DIMENSION = 3072

In [7]:
azure_openai_client = AzureOpenAI(api_version=AZURE_OPENAI_API_VERSION, azure_endpoint=AZURE_OPENAI_ENDPOINT, api_key=AZURE_OPENAI_API_KEY)

In [ ]:
sql_files = []
if DATABASE_FILES_DIR.exists() and DATABASE_FILES_DIR.is_dir():
    for sql_file in DATABASE_FILES_DIR.rglob("*.sql"):
        with open(sql_file, encoding="utf-8") as f:
            content = f.read()

        sql_files.append((sql_file, content))

In [8]:
system_prompt = """
You are an expert database documentation specialist. Your role is to create comprehensive, detailed documentation for database objects based on DDL (Data Definition Language) statements provided by users.

## Core Instructions

When a user provides DDL for any database object, create thorough documentation that extracts and utilizes ALL available information including comments, constraints, metadata, and specifications. Provide the most complete documentation possible - no detail is too small.

## Documentation Structure

Generate documentation with these sections:

### Object Overview
- Identify the type of database object (table, view, procedure, function, index, trigger, etc.)
- Explain the primary purpose and role within the database schema
- Provide business context and main use cases

### Detailed Structure & Components
- **For tables/views:** Document every column with complete details from DDL comments and constraints
- **For procedures/functions:** Detail all parameters, return types, and logic flow
- **For indexes:** Specify all columns covered, index type, purpose, and performance impact
- **For other objects:** Cover all relevant structural elements with full specifications

### Component Analysis (Leverage ALL DDL Comments)
- Extract business meaning and purpose from inline comments
- Document complete data type specifications (precision, scale, length)
- List all validation rules, constraints, and business logic
- Identify required vs optional elements with reasoning
- Explain default values, their significance, and business rationale
- Note any special handling, edge cases, or implementation details

### Complete Relationship Mapping
- Map all foreign key relationships with detailed explanations
- Identify self-referencing relationships and hierarchical structures
- Document dependencies on other database objects
- List objects that depend on this one
- Provide impact analysis for changes or cascading operations

### Comprehensive Constraints & Rules
- Document every constraint with business justification
- List all business rules enforced at database level
- Note security, access, and data integrity considerations
- Explain performance implications and optimization details

### Usage Patterns & Integration
- Describe how the object fits into larger business processes
- Detail common and advanced interaction patterns
- Identify query patterns this object supports
- Note performance characteristics and tuning considerations
- Explain integration points with applications

### Implementation Details
- Document storage specifications and logging settings
- Note any special database features utilized
- Include maintenance and operational considerations

## Quality Standards

- **Comprehensiveness:** Extract every piece of information from the DDL
- **Clarity:** Use clear, professional language accessible to technical and business users
- **Structure:** Organize with clear markdown headings for easy scanning
- **Accuracy:** Base all documentation strictly on the provided DDL
- **Detail:** Include all constraints, comments, data types, and specifications

## Output Format

Provide well-structured markdown documentation with clear headings. Make the documentation comprehensive yet scannable, suitable for database developers, business analysts, application developers, data architects, and database administrators.

Always begin your response with a clear heading identifying the database object name and type.
"""

In [ ]:
for md_file in DATABASE_FILES_DIR.rglob("*.md"):
    try:
        md_file.unlink(missing_ok=True)
    except PermissionError:
        logging.error(f"Permission denied deleting file {md_file}")
    except Exception as e:
        logging.error(f"Unexpected error deleting file {md_file}: {e}")

In [ ]:
for sql_file, content in sql_files:
    user_prompt = content.strip()
    messages = []
    messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_prompt})
    chat_completion = azure_openai_client.chat.completions.create(
            model=AZURE_OPENAI_DEPLOYMENT,
            messages=messages,
            temperature=0,
            top_p=1,
            n=1,
        ).choices[0].message.content
    with open(sql_file.with_suffix(".md"), "w") as f:
        f.write(chat_completion)


In [ ]:
for md_file in DATABASE_FILES_DIR.rglob("*.md"):
    try:
        target_file = Path(WORKING_DIR) / md_file.name
        target_file.write_text(md_file.read_text(encoding="utf-8"), encoding="utf-8")
    except PermissionError:
        logging.error(f"Permission denied copying file {md_file} to {target_file}")
    except Exception as e:
        logging.error(f"Unexpected error copying file {md_file} to {target_file}: {e}")

In [14]:
async def llm_model_func(
    prompt, system_prompt=None, history_messages=[], keyword_extraction=False, **kwargs
) -> str:
    client = AzureOpenAI(
        api_key=AZURE_OPENAI_API_KEY,
        api_version=AZURE_OPENAI_API_VERSION,
        azure_endpoint=AZURE_OPENAI_ENDPOINT,
    )

    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    if history_messages:
        messages.extend(history_messages)
    messages.append({"role": "user", "content": prompt})

    chat_completion = client.chat.completions.create(
        model=AZURE_OPENAI_DEPLOYMENT,
        messages=messages,
        temperature=kwargs.get("temperature", 0),
        top_p=kwargs.get("top_p", 1),
        n=kwargs.get("n", 1),
    )
    return chat_completion.choices[0].message.content

In [15]:
async def embedding_func(texts: list[str]) -> np.ndarray:
    client = AzureOpenAI(
        api_key=AZURE_OPENAI_API_KEY,
        api_version=AZURE_EMBEDDING_API_VERSION,
        azure_endpoint=AZURE_OPENAI_ENDPOINT,
    )
    embedding = client.embeddings.create(model=AZURE_EMBEDDING_DEPLOYMENT, input=texts)

    embeddings = [item.embedding for item in embedding.data]
    return np.array(embeddings)

In [16]:
async def initialize_rag():
    rag = LightRAG(
        working_dir=WORKING_DIR.name,
        llm_model_func=llm_model_func,
        embedding_func=EmbeddingFunc(
            embedding_dim=EMBEDDING_DIMENSION,
            max_token_size=8192,
            func=embedding_func,
        ),
    )

    await rag.initialize_storages()
    await initialize_pipeline_status()

    return rag

In [17]:
nest_asyncio.apply()

rag = await initialize_rag()

INFO:nano-vectordb:Load (931, 3072) data
INFO:nano-vectordb:Init {'embedding_dim': 3072, 'metric': 'cosine', 'storage_file': 'working_dir/vdb_entities.json'} 931 data
INFO:nano-vectordb:Load (1717, 3072) data
INFO:nano-vectordb:Init {'embedding_dim': 3072, 'metric': 'cosine', 'storage_file': 'working_dir/vdb_relationships.json'} 1717 data
INFO:nano-vectordb:Load (40, 3072) data
INFO:nano-vectordb:Init {'embedding_dim': 3072, 'metric': 'cosine', 'storage_file': 'working_dir/vdb_chunks.json'} 40 data
Rerank is enabled but no rerank_model_func provided. Reranking will be skipped.


In [ ]:
for md_file in WORKING_DIR.rglob("*.md"):
    print(md_file.relative_to(WORKING_DIR))
    with open(md_file, encoding="utf-8") as doc:
        rag.insert([doc.read()])

In [18]:
query_text = "What tables the emp_details_view uses?"

In [19]:
print("Result (Naive):")
print(rag.query(query_text, param=QueryParam(mode="naive")))

Result (Naive):


Rerank is enabled but no rerank model is configured. Please set up a rerank model or set enable_rerank=False in query parameters.


## Tables Used by `hr.EMP_DETAILS_VIEW`

The `hr.EMP_DETAILS_VIEW` is a read-only view in the HR schema that consolidates employee information by joining several related tables. Specifically, it is constructed from the following base tables:

1. **hr.employees** (aliased as `e`): Contains core employee data such as employee ID, name, job, manager, salary, and department assignment.
2. **hr.departments** (aliased as `d`): Provides department details, including department name and location.
3. **hr.jobs** (aliased as `j`): Supplies job-related information like job title.
4. **hr.locations** (aliased as `l`): Contains location data for departments, including city and state/province.
5. **hr.countries** (aliased as `c`): Offers country information for each location.
6. **hr.regions** (aliased as `r`): Provides regional data associated with each country.

These tables are joined together to present a comprehensive, denormalized snapshot of each employee, including their organizational and g

INFO:openai._base_client:Retrying request to /chat/completions in 28.000000 seconds
INFO:openai._base_client:Retrying request to /chat/completions in 40.000000 seconds
INFO:openai._base_client:Retrying request to /chat/completions in 35.000000 seconds
INFO:openai._base_client:Retrying request to /chat/completions in 46.000000 seconds


In [20]:
print("\nResult (Local):")
print(rag.query(query_text, param=QueryParam(mode="local")))


Result (Local):


Rerank is enabled but no rerank model is configured. Please set up a rerank model or set enable_rerank=False in query parameters.


## Tables Used by `hr.EMP_DETAILS_VIEW`

The `hr.EMP_DETAILS_VIEW` is a comprehensive, read-only view in the HR schema that consolidates employee, job, department, and geographic data for reporting and analysis. It is constructed by joining several underlying tables to provide a denormalized snapshot of employee details.

### Underlying Tables

The view uses the following tables:

1. **hr.employees**  
   Contains detailed records for each employee, including identifiers, names, job assignments, salaries, and manager relationships.

2. **hr.departments**  
   Stores information about organizational departments, including department names and location assignments.

3. **hr.jobs**  
   Defines all valid job roles and their associated salary ranges, used to assign job titles and codes to employees.

4. **hr.locations**  
   Contains information about the physical locations of departments, such as city, state, and country codes.

5. **hr.countries**  
   Provides country codes and names, m

In [21]:
print("\nResult (Global):")
print(rag.query(query_text, param=QueryParam(mode="global")))


Result (Global):


Rerank is enabled but no rerank model is configured. Please set up a rerank model or set enable_rerank=False in query parameters.


## Tables Used by `hr.EMP_DETAILS_VIEW`

The `hr.EMP_DETAILS_VIEW` is a read-only view in the HR schema designed to provide a comprehensive, denormalized snapshot of employee details. It achieves this by joining multiple related tables within the HR schema. The specific tables used by this view are:

- **hr.employees**: Contains detailed records for each employee, including job, department, and manager information.
- **hr.departments**: Stores information about each department, including department ID and location ID.
- **hr.jobs**: Defines valid job roles and titles that can be assigned to employees.
- **hr.locations**: Contains data about physical locations, including city, state/province, and country ID.
- **hr.countries**: Provides country codes and names, supporting geographic context.
- **hr.regions**: Defines regions for further geographic segmentation.

These tables are joined using foreign key relationships to ensure data integrity and to provide a unified view of employee, or

In [22]:
print("\nResult (Hybrid):")
print(rag.query(query_text, param=QueryParam(mode="hybrid")))


Result (Hybrid):


Rerank is enabled but no rerank model is configured. Please set up a rerank model or set enable_rerank=False in query parameters.


## Tables Used by `hr.EMP_DETAILS_VIEW`

The `hr.EMP_DETAILS_VIEW` is a read-only view in the HR schema that consolidates employee and organizational data for comprehensive reporting and analysis. It is constructed by joining several core tables within the HR schema. Specifically, the view uses the following tables:

1. **hr.employees**  
   Stores detailed records for each employee, including identifiers, names, job assignments, salaries, manager relationships, and department assignments.

2. **hr.departments**  
   Contains information about each organizational department, including department names and associated locations.

3. **hr.jobs**  
   Defines all valid job roles and their associated salary ranges, providing job titles and codes for employees.

4. **hr.locations**  
   Stores location data for departments, including city, state/province, and country identifiers.

5. **hr.countries**  
   Provides country information, mapping locations to their respective countries.

6. **hr

In [23]:
print("\nResult (Hybrid):")
print(rag.query("list all the objects that use the departments table, just list them, do not add any other information", param=QueryParam(mode="hybrid")))


Result (Hybrid):


Rerank is enabled but no rerank model is configured. Please set up a rerank model or set enable_rerank=False in query parameters.


- HR.EMPLOYEES table  
- HR.JOB_HISTORY table  
- HR.LOCATIONS table  
- hr.EMP_DETAILS_VIEW  
- EMP_DEPARTMENT_IX (Index)  
- Payroll and finance systems  
- HR applications  
- Reporting and analytics objects  

References:
- [KG] hr.employees.md
- [KG] hr.job_history.md
- [KG] hr.locations.md
- [KG] hr.emp_details_view.md
- [DC] hr.emp_department_ix.md


In [24]:
print("\nResult (Hybrid):")
print(rag.query("how does the LOCATIONS table uses the departments table?", param=QueryParam(mode="hybrid")))


Result (Hybrid):


Rerank is enabled but no rerank model is configured. Please set up a rerank model or set enable_rerank=False in query parameters.


## Relationship Between LOCATIONS and DEPARTMENTS Tables

### Overview

In the HR schema, the relationship between the `LOCATIONS` and `DEPARTMENTS` tables is primarily one of reference and association, but it is important to clarify the direction of this relationship:

- The `DEPARTMENTS` table references the `LOCATIONS` table, not the other way around.
- Each department can be associated with a specific physical location through the `LOCATION_ID` foreign key in the `DEPARTMENTS` table.

### How the Relationship Works

- **Foreign Key in DEPARTMENTS:**  
  The `DEPARTMENTS` table contains a `LOCATION_ID` column, which is a foreign key referencing the primary key (`LOCATION_ID`) in the `LOCATIONS` table. This ensures that any location assigned to a department must exist in the `LOCATIONS` table, maintaining referential integrity.
- **Purpose:**  
  This setup allows each department to be linked to a valid location, supporting business processes such as resource allocation, reporting, a

In [ ]:
print("\nResult (Hybrid):")
print(rag.query("write a query that list all the employees by locations and countries. required field: location id, county id, employee id", param=QueryParam(mode="hybrid")))

In [42]:
# create a Path object to the graphml file in the working directory
graphml_file = WORKING_DIR / "graph_chunk_entity_relation.graphml"

In [43]:
G = nx.read_graphml(graphml_file)
net = Network(height="600px", width="100%", bgcolor="#ffffff", font_color="black", 
              notebook=True, cdn_resources='in_line')
net.from_nx(G)

In [44]:
for node in net.nodes:
    node["color"] = "#{:06x}".format(random.randint(0, 0xFFFFFF))
    if "description" in node:
        node["title"] = node["description"]

In [45]:
for edge in net.edges:
    if "description" in edge:
        edge["title"] = edge["description"]

In [51]:
net.show("knowledge_graph.html")

knowledge_graph.html
